# The pin photodiode {#sec-pin}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/pin.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/pin.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*Slides 29–30 (structure, field and capacitance of the pin photodiode), 31 (drift velocity in Si), 32 (diffusion of carriers
generated in the p⁺ layer), 37–40 (worked examples: transit time, diffusion time) and 41–43 (steady-state photocurrent).*

## Key relations

A pn junction has a narrow depletion region whose width depends on the bias (Ch. 3), so most light is absorbed in the neutral
regions and collected slowly by diffusion. The **pin** photodiode inserts an intrinsic (lightly doped) layer of width $W$ between
p⁺ and n⁺ (slide 29). The depletion region is the whole i-layer, nearly independent of the bias, with a uniform field
(slide 30):

$$
E=E_o+\frac{V_r}{W}\approx\frac{V_r}{W},\qquad C_\text{dep}=\frac{\varepsilon_o\varepsilon_rA}{W},\qquad
t_\text{drift}=\frac{W}{v_d}.
$$

The drift velocity saturates at high field, about $10^5$ m/s for electrons in Si above $\sim3\times10^6$ V/m (slide 31), so the
transit time cannot be reduced indefinitely by raising $V_r$. Carriers generated in the p⁺ layer outside the field must diffuse
to the i-layer; in a time $t$ they diffuse on average (slide 32)

$$
\ell=\sqrt{2D_et}\quad\Longrightarrow\quad t_\text{diff}=\frac{\ell^2}{2D_e},
$$

which is much slower than drift and produces the slow **diffusion tail** of the pulse response (@sec-circuits).

The pin therefore embodies the central trade-off of photodetection: a wide $W$ absorbs more light (high $\mathcal R$, @sec-responsivity)
and has a small capacitance (small $RC$), but a long transit time. A narrow $W$ is fast in transit but absorbs little and has a large
capacitance. For a given load there is an **optimum width**.

## Drift velocity in silicon (slide 31)

In [ ]:
#| label: fig-vd
#| fig-cap: "Electron and hole drift velocities in Si at 300 K (Caughey–Thomas model with the parameters of Jacoboni et al.), cf. slide 31. Dotted lines: the fields of the worked examples of slides 37 (5×10⁶ V/m) and 39 (3×10⁶ V/m)."
E = np.logspace(4, 7.3, 300)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.loglog(E, dl.drift_velocity(E, "e"), label="electrons"); ax.loglog(E, dl.drift_velocity(E, "h"), label="holes")
for x in (5e6, 3e6): ax.axvline(x, color=dl.INK2, ls=":", lw=1)
ax.set(xlabel="electric field (V/m)", ylabel="drift velocity (m/s)", ylim=(1e2, 2e5)); ax.legend(); plt.show()

## Worked examples (slides 37–43), recomputed

> **Slides 37–38.** Si pin, $W=20$ µm, thin p⁺ (0.1 µm), $V_r=100$ V, short pulse at 900 nm absorbed over the whole i-layer:
> duration of the photocurrent? **Slides 39–40.** Same pin at 60 V, light absorbed at the surface of a 1 µm p⁺ layer,
> $D_e=3\times10^{-4}$ m² s⁻¹: response time? **Slides 42–43.** Steady state at 900 nm, 100 nW, no AR coating, $W=20$ µm;
> then with a perfect AR coating; then with $W=40$ µm.

In [ ]:
#| label: tbl-pin
#| tbl-cap: "Worked examples of slides 37–43. Velocities from the Caughey–Thomas model rather than read from the graph."
W = 20e-6
E1 = 100/W; ve1, vh1 = dl.drift_velocity(E1, "e"), dl.drift_velocity(E1, "h")
E2 = 60/W
tdiff = dl.diffusion_time(1e-6, 3e-4)
T = dl.fresnel_T(3.6)
I0 = dl.pin_photocurrent(100e-9, 900e-9, 3e4, 20e-6, T)
I1 = dl.pin_photocurrent(100e-9, 900e-9, 3e4, 20e-6, 1.0)
I2 = dl.pin_photocurrent(100e-9, 900e-9, 3e4, 40e-6, 1.0)
rows = [("s37: α(900 nm), absorption depth", f"{dl.alpha('Si', 900e-9):.2e} m⁻¹, {1e6/dl.alpha('Si', 900e-9):.0f} µm", "3×10⁴ m⁻¹, 33 µm"),
        ("s37: field V_r/W", f"{E1:.1e} V/m", "5×10⁶ V/m"),
        ("s38: v_e, v_h at 5×10⁶ V/m", f"{ve1:.2e}, {vh1:.2e} m/s", "≈10⁵, 7×10⁴ m/s"),
        ("s38: hole transit time W/v_h", f"{W/vh1*1e9:.3f} ns", "0.29 ns"),
        ("s40: diffusion time across 1 µm p⁺", f"{tdiff*1e9:.2f} ns", "1.67 ns"),
        ("s40: electron drift time at 3×10⁶ V/m", f"{W/dl.drift_velocity(E2, 'e')*1e9:.2f} ns (v_sat: {W/1e5*1e9:.2f} ns)", "0.2 ns"),
        ("s40: total, t_diff + t_drift", f"{(tdiff + W/1e5)*1e9:.2f} ns", "1.87 ns"),
        ("s43: transmission of bare Si, T", f"{T:.3f}", "0.68"),
        ("s43: I_ph, no AR coating", f"{I0*1e9:.1f} nA, R = {I0/100e-9:.3f} A/W", "22 nA, 0.22 A/W"),
        ("s43: I_ph, perfect AR coating", f"{I1*1e9:.1f} nA, R = {I1/100e-9:.3f} A/W", "32.7 nA, 0.33 A/W"),
        ("s43: W = 40 µm, perfect AR", f"R = {I2/100e-9:.3f} A/W", "0.51 A/W"),
        ("s43: upper limit (η = 1)", f"{dl.responsivity(1, 900e-9):.3f} A/W", "0.73 A/W")]
pd.DataFrame(rows, columns=["quantity", "computed", "slide"])

The model gives $v_e=0.89\times10^5$ m/s at $3\times10^6$ V/m, slightly below the "saturated $10^5$ m/s" of slide 40: saturation is
gradual. The main lesson of slides 39–40 is untouched: **one micrometre of diffusion costs eight times more than twenty
micrometres of drift.**

## Live panel: designing a fast Si pin (transit time versus RC)

The bandwidth combines the transit-time response of @sec-ramo (generation $e^{-\alpha x}$ in the i-layer, electrons and holes
at their field-dependent velocities) with the $RC$ low-pass of the junction capacitance and the load (@sec-circuits):
$|H(f)|=|H_\text{transit}(f)|\,/\,|1+j2\pi fRC|$.

```{ojs}
//| echo: false
import {alpha, driftVelocity, transitMag, responsivity, fresnelT, eps0} from "./det_sim.js"
viewof pn2 = Inputs.form({
  wl: Inputs.range([400, 1050], {value: 850, step: 5, label: "wavelength λ (nm)"}),
  W: Inputs.range([0.5, 100], {value: 20, step: 0.5, label: "i-layer width W (µm)"}),
  Vr: Inputs.range([1, 200], {value: 20, step: 1, label: "reverse bias V_r (V)"}),
  D: Inputs.range([10, 3000], {value: 300, step: 10, label: "active diameter (µm)"}),
  RL: Inputs.range([10, 10000], {value: 50, step: 10, label: "load resistance R_L (Ω)"}),
  ar: Inputs.toggle({label: "perfect AR coating", value: true})
})
```

```{ojs}
//| echo: false
pinOut = {
  const a = alpha("Si", pn2.wl * 1e-9), A = Math.PI * (pn2.D * 1e-6 / 2) ** 2, T = pn2.ar ? 1 : fresnelT(3.6);
  const bw = (Wum) => {
    const W = Wum * 1e-6, E = pn2.Vr / W, ve = driftVelocity(E, "e"), vh = driftVelocity(E, "h");
    const C = eps0 * 11.9 * A / W, fRC = 1 / (2 * Math.PI * pn2.RL * C);
    const ftr0 = 0.45 * Math.min(ve, vh) / W, fmax = 3 * Math.min(ftr0 * 2, fRC * 2);
    let f3 = NaN, ftr = NaN;
    for (let i = 1; i <= 600; i++) { const f = fmax * i / 600, Ht = transitMag(f, W, a, ve, vh, 60);
      if (isNaN(ftr) && Ht < Math.SQRT1_2) ftr = f;
      if (Ht / Math.hypot(1, f / fRC) < Math.SQRT1_2) { f3 = f; break; } }
    return {W: Wum, f3, fRC, ftr, ve, vh, C, E, R: responsivity(T, pn2.wl * 1e-9) * (1 - Math.exp(-a * W))};
  };
  const cur = bw(pn2.W), sweep = d3.range(0, 61).map(i => bw(0.5 * 200 ** (i / 60)));
  const best = sweep.reduce((b, s) => (s.f3 > b.f3 ? s : b), sweep[0]);
  const bestRB = sweep.reduce((b, s) => (s.f3 * s.R > b.f3 * b.R ? s : b), sweep[0]);
  return {cur, sweep, best, bestRB, a};
}
Plot.plot({
  width: Math.min(width, 1050), height: 300, x: {label: "i-layer width W (µm)", type: "log"},
  y: {label: "−3 dB bandwidth (GHz)", type: "log", grid: true},
  color: {legend: true, domain: ["total", "transit only", "RC only"], range: ["#0b0b0b", "#2a78d6", "#e34948"]},
  marks: [Plot.line(pinOut.sweep, {x: "W", y: d => d.f3 / 1e9, stroke: () => "total", strokeWidth: 2.5}),
          Plot.line(pinOut.sweep, {x: "W", y: d => d.ftr / 1e9, stroke: () => "transit only", strokeDasharray: "5,3"}),
          Plot.line(pinOut.sweep, {x: "W", y: d => d.fRC / 1e9, stroke: () => "RC only", strokeDasharray: "5,3"}),
          Plot.dot([pinOut.cur], {x: "W", y: d => d.f3 / 1e9, r: 6, fill: "#1baf7a"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">
W = ${pn2.W} µm: field ${(pinOut.cur.E / 1e5).toFixed(1)} kV/cm, v<sub>e</sub> = ${(pinOut.cur.ve / 1e3).toFixed(0)} km/s, v<sub>h</sub> = ${(pinOut.cur.vh / 1e3).toFixed(0)} km/s,
C = ${(pinOut.cur.C * 1e12).toFixed(2)} pF<br>
f<sub>transit</sub> = ${(pinOut.cur.ftr / 1e9).toFixed(2)} GHz · f<sub>RC</sub> = ${(pinOut.cur.fRC / 1e9).toFixed(2)} GHz ·
<b>f<sub>3dB</sub> = ${(pinOut.cur.f3 / 1e9).toFixed(2)} GHz</b> · responsivity at ${pn2.wl} nm: <b>${pinOut.cur.R.toFixed(3)} A/W</b>
(1/α = ${(1e6 / pinOut.a).toPrecision(3)} µm)<br>
fastest width ${pinOut.best.W.toFixed(1)} µm (${(pinOut.best.f3 / 1e9).toFixed(2)} GHz, R = ${pinOut.best.R.toFixed(2)} A/W) ·
best R × bandwidth at ${pinOut.bestRB.W.toFixed(1)} µm (${(pinOut.bestRB.f3 / 1e9).toFixed(2)} GHz, R = ${pinOut.bestRB.R.toFixed(2)} A/W)</div>`
```

Things to try: with a 300 µm detector on 50 Ω the RC limit dominates below about 9 µm and the transit time above; the optimum,
near 9 µm, gives 2.6 GHz but only 0.26 A/W at 850 nm. A 1 mm detector is optimal near 22 µm and 0.6 GHz. Shrink the diameter to
30 µm (a fiber-coupled datacom detector): the optimum moves to about 1 µm and 30 GHz, but Si then absorbs less than 5 % at 850 nm. That is why 25 Gb/s receivers at 850 nm use GaAs (α 20 times larger) and why 1310/1550 nm receivers
use InGaAs layers about 1 µm thick. Raise $V_r$ at small $W$: nothing changes once the velocities saturate.

In [ ]:
#| label: fig-pin-tradeoff
#| fig-cap: "Bandwidth–efficiency trade-off of Si pin photodiodes at 850 nm on a 50 Ω load (V_r = 20 V, perfect AR coating) for three diameters. Solid: −3 dB bandwidth (transit × RC); dashed: responsivity. Small detectors can be thin and fast, at the cost of responsivity."
Ws = np.logspace(np.log10(0.5e-6), np.log10(100e-6), 40)
fig, ax = plt.subplots(figsize=(7.5, 3.8)); ax2 = ax.twinx()
for Dm, c in zip((30e-6, 100e-6, 300e-6), dl.PALETTE):
    A = np.pi*(Dm/2)**2
    f = [dl.pin_bandwidth(W, A, 50, dl.alpha("Si", 850e-9), dl.drift_velocity(20/W, "e"), dl.drift_velocity(20/W, "h"))[0] for W in Ws]
    ax.loglog(Ws*1e6, np.array(f)/1e9, color=c, label=f"⌀ {Dm*1e6:.0f} µm")
ax2.semilogx(Ws*1e6, dl.pin_photocurrent(1, 850e-9, dl.alpha("Si", 850e-9), Ws), "--", color=dl.INK2)
ax2.set_ylabel("R at 850 nm (A/W), dashed"); ax2.grid(False)
ax.set(xlabel="W (µm)", ylabel="f₃dB (GHz)"); ax.legend(fontsize=8, loc="lower center"); plt.show()

::: {.callout-tip collapse="true"}
## Going further
- **The optimum width analytically.** With a single carrier at saturation, $f_\text{tr}\approx0.45v_s/W$ and $f_{RC}=W/(2\pi R\varepsilon A)$.
  Setting $f_\text{tr}=f_{RC}$ gives $W_\text{opt}=\sqrt{0.9\pi R\varepsilon Av_s}$ and a maximum bandwidth $\propto1/\sqrt{A}$. Check this
  against the panel.
- **Velocity overshoot and hot carriers** in InGaAs/InP let electrons cross thin layers at 3–5×10⁵ m/s, faster than the
  equilibrium saturation velocity. Combine with the UTC structure of @sec-ramo.
- **Inductive peaking.** A small series inductance (bond wire) can extend the RC bandwidth by up to 70 % at the cost of ringing.
  Simulate an RLC low-pass with $L=0.5$ nH, $C=0.3$ pF, $R=50$ Ω.
- **Silicon for 1550 nm?** Ge-on-Si waveguide photodiodes are now standard in silicon photonics transceivers: Ge grown on Si
  absorbs to about 1.6 µm thanks to tensile strain, which lowers its direct gap.
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 5.5; Sze & Ng, sec. 13.3.
- K. Kato, "Ultrawide-band/high-frequency photodetectors", *IEEE Trans. Microw. Theory Tech.* 47, 1265 (1999): transit-time/RC
  trade-off, waveguide and travelling-wave photodiodes.
- C. Jacoboni *et al.*, "A review of some charge transport properties of silicon", *Solid-State Electron.* 20, 77 (1977): the
  velocity–field data used here.
:::

## Try it yourself

1. Redo slide 37 for 850 nm instead of 900 nm. Is the assumption "absorption over the whole i-layer" better or worse?
2. In slide 39 the p⁺ layer is 1 µm thick. How thin must it be for the diffusion time to equal the 0.2 ns drift time?
3. Compute the capacitance of the slide-37 pin for a 1 mm diameter and the bandwidth with a 50 Ω load. Which limit dominates?
4. Explain why increasing $V_r$ from 10 V to 100 V reduces the capacitance of a pn photodiode but hardly changes that of a pin.